<a href="https://colab.research.google.com/github/bahmedx/730/blob/main/MCMC_Use_in_Industry.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Step 1: Install Packages**

In [ ]:
!pip -q install pymc arviz

import pandas as pd
import numpy as np
import pymc as pm
import arviz as az

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

np.random.seed(42)
print("Libraries loaded successfully.")

# **Step 2: Upload Dataset**

In [ ]:
# Upload Dataset
import zipfile
from google.colab import files

uploaded = files.upload()

zip_file = list(uploaded.keys())[0]

extract_dir = "heart_data"

with zipfile.ZipFile(zip_file, "r") as zip_ref:
  zip_ref.extractall(extract_dir)

print("Dataset extracted successfully.")

# **Step 3: Data Extraction and Loading**

In [ ]:
from zipfile import ZipFile

# Data Extraction and Loading
zip_file_path = "Assignment_4_Lab_1_DataSet_statlog+german+credit+data.zip"
data_file_name = "german.data"

# Extract and load the dataset
with ZipFile(zip_file_path, 'r') as z:
    with z.open(data_file_name) as f:

        # The dataset has 20 attributes; index 20 is the risk label
        df = pd.read_csv(
            f,
            sep=' ',
            header=None
        )

# Capture original dataset characteristics
total_observations = df.shape[0]
original_predictor_variables = df.shape[1] - 1

print(
    f"Dataset loaded successfully with shape: {df.shape}"
)

df.head()

### **Step 3A: Load Dataset Documentation (Silent)**

In [ ]:
# ==================================================
# Step 3A: Load Dataset Documentation (Silent)
# ==================================================

from zipfile import ZipFile
import re

code_map = {}

with ZipFile(
    "Assignment_4_Lab_1_DataSet_statlog+german+credit+data.zip",
    "r"
) as z:

    german_doc = z.read(
        "german.doc"
    ).decode(
        "latin-1",
        errors="ignore"
    )

    for line in german_doc.splitlines():

        line = line.strip()

        match = re.match(
            r"(A\d+)\s*:\s*(.*)",
            line
        )

        if match:
            code_map[
                match.group(1)
            ] = match.group(2)

print(
    f"Documentation loaded successfully. "
    f"{len(code_map)} category descriptions identified."
)

### **Step 3B: Create Dynamic Code Mapping**

In [ ]:
# ==================================================
# Step 3B: Create Dynamic Code Mapping
# ==================================================

import re

code_map = {}

for line in german_doc.splitlines():

    line = line.strip()

    match = re.match(
        r"(A\d+)\s*:\s*(.*)",
        line
    )

    if match:
        code_map[
            match.group(1)
        ] = match.group(2)

print(f"Mappings Found: {len(code_map)}")

list(code_map.items())[:10]

# **Step 4: Data Preprocessing**

In [ ]:
# Data Preprocessing
# Target transformation: 1 (Good) -> 0 (Low Risk), 2 (Bad) -> 1 (High Risk)
y = df.iloc[:, 20].apply(lambda x: 0 if x == 1 else 1).values

# Feature selection: Duration (col 1) and Credit Amount (col 4)
# Note: In a real supply chain scenario, these could be 'Lead Time' and 'Order Volume'
X = df.iloc[:, [1, 4]].values

# Standardize features (Z-score normalization) for better MCMC convergence
X_standardized = (X - X.mean(axis=0)) / X.std(axis=0)

print("Data preprocessing complete. Features standardized.")

In [ ]:
df = pd.get_dummies(df, drop_first=True)

y = df.iloc[:, -1]

# Convert target
y = (y == 1).astype(int)

X = df.iloc[:, :-1]
# Convert column names to strings to resolve TypeError with StandardScaler
X.columns = X.columns.astype(str)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print("Features:", X.shape)
print("Target:", y.shape)
print("Training:", X_train.shape)
print("Testing:", X_test.shape)

In [ ]:
# ==================================================
# Dataset Characteristics Summary
# ==================================================

dataset_summary = pd.DataFrame({
    "Characteristic": [
        "Total Observations",
        "Original Predictor Variables",
        "Encoded Predictor Variables",
        "Training Records",
        "Testing Records"
    ],
    "Value": [
        total_observations,
        original_predictor_variables,
        X.shape[1],
        X_train.shape[0],
        X_test.shape[0]
    ]
})

display(
    dataset_summary.style
    .hide(axis="index")
    .set_caption(
        "Dataset Characteristics"
    )
)

### **Step 4A: Class Distribution Analysis**

In [ ]:
# ==================================================
# Step 4A: Class Distribution Summary
# ==================================================

train_counts = pd.Series(y_train).value_counts().sort_index()

test_counts = pd.Series(y_test).value_counts().sort_index()

train_pct = (
    pd.Series(y_train)
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)
)

test_pct = (
    pd.Series(y_test)
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)
)

# Create DataFrame FIRST
class_summary = pd.DataFrame({
    "Training Count": train_counts,
    "Training %": train_pct,
    "Testing Count": test_counts,
    "Testing %": test_pct
})

# Replace index labels with meaningful names
class_summary.index = [
    "Low Risk (0)",
    "High Risk (1)"
]

# Convert index to a column
class_summary = class_summary.reset_index()

class_summary.columns = [
    "Risk Class",
    "Training Count",
    "Training %",
    "Testing Count",
    "Testing %"
]

display(
    class_summary.style
    .hide(axis="index")
    .format({
        "Training %": "{:.2f}%",
        "Testing %": "{:.2f}%"
    })
    .set_caption(
        "Class Distribution of Risk Categories"
    )
)

# **Step 5: Bayesian Logistic Regression (MCMC)**

In [ ]:
with pm.Model() as bayesian_supply_chain:

    beta = pm.Normal(
        'beta',
        mu=0,
        sigma=2,
        shape=X_train.shape[1]
    )

    intercept = pm.Normal(
        'intercept',
        mu=0,
        sigma=2
    )

    logits = intercept + pm.math.dot(X_train, beta)

    p = pm.Deterministic(
        "p",
        pm.math.sigmoid(logits)
    )

    positive_weight = (
    len(y_train) /
    (2 * y_train.sum())
    )

    sample_weights = np.where(
        y_train == 1,
        positive_weight,
        1
    )

    y_obs = pm.Potential(
        "weighted_likelihood",
        sample_weights *
        pm.logp(
            pm.Bernoulli.dist(p=p),
            y_train
        )
    )

    trace = pm.sample(
        draws=2000,
        tune=1000,
        chains=4,
        target_accept=0.95,
        random_seed=42
    )

# **Step 6: Convergence Diagnostics**

In [ ]:
# Result Analysis and Visualization
# Plot the distributions and sampling paths
az.plot_trace(trace, var_names=["intercept", "beta"])

# Generate statistical summary
summary = az.summary(trace, var_names=["intercept", "beta"])
print("\n--- MCMC Model Posterior Summary ---")
display(summary)

In [ ]:
fig, ax = plt.subplots(
    figsize=(6, 3)
)

az.plot_posterior(
    trace,
    var_names=["intercept"],
    ax=ax
)

plt.tight_layout()
plt.show()

# **Step 7: Prediction**

In [ ]:
posterior_beta = trace.posterior["beta"].mean(
    dim=("chain","draw")
).values

posterior_intercept = trace.posterior["intercept"].mean().values

logits_test = (
    posterior_intercept +
    np.dot(X_test, posterior_beta)
)

probability = 1 / (1 + np.exp(-logits_test))

prediction = (probability > 0.5).astype(int)

print("First 10 Probabilities")

pd.DataFrame({
    "Probability": probability[:10],
    "Prediction": prediction[:10]
})

# **Step 8: Model Performance and Classification Report**

In [ ]:
accuracy = accuracy_score(
    y_test,
    prediction
)

roc = roc_auc_score(
    y_test,
    probability
)

print("Accuracy:", accuracy)
print("ROC-AUC:", roc)

print(
    classification_report(
        y_test,
        prediction
    )
)

# **Step 9: Confusion Matrix**

In [ ]:
# ==========================================
# Confusion Matrix and ROC Curve
# ==========================================

from sklearn.metrics import RocCurveDisplay

fig, axes = plt.subplots(
    1, 2,
    figsize=(8, 3)
)

# -----------------------
# Confusion Matrix
# -----------------------
cm = confusion_matrix(
    y_test,
    prediction
)

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    ax=axes[0]
)

axes[0].set_title(
    "Confusion Matrix"
)

axes[0].set_xlabel(
    "Predicted"
)

axes[0].set_ylabel(
    "Actual"
)

# -----------------------
# ROC Curve
# -----------------------
RocCurveDisplay.from_predictions(
    y_test,
    probability,
    ax=axes[1]
)

axes[1].set_title(
    f"ROC Curve (AUC = {roc:.3f})"
)

plt.tight_layout()
plt.show()

In [ ]:
# ==========================================
# Precision Recall Curve
# ==========================================

from sklearn.metrics import (
    PrecisionRecallDisplay,
    average_precision_score
)

ap = average_precision_score(
    y_test,
    probability
)

fig, ax = plt.subplots(
    figsize=(7, 3.5)
)

PrecisionRecallDisplay.from_predictions(
    y_test,
    probability,
    ax=ax
)

ax.set_title(
    "Precision-Recall Curve"
)

ax.legend(
    [f"Classifier (AP = {ap:.3f})"],
    loc="center left",
    bbox_to_anchor=(1.02, 0.5),
    frameon=True
)

plt.tight_layout()
plt.show()

# **Step 10: Posterior Feature Importance**

In [ ]:
# ==================================================
# Step 10: Posterior Feature Importance
# ==================================================

importance = pd.DataFrame({
    "Feature": X.columns,
    "Posterior_Mean": posterior_beta
})

importance = (
    importance
    .sort_values(
        "Posterior_Mean",
        ascending=False
    )
    .reset_index(drop=True)
)

# Decode labels using german.doc mappings
def decode_feature(feature):

    feature = str(feature)

    # Continuous variables
    if "_" not in feature:
        return feature

    # Categorical dummy variables
    _, code = feature.split("_", 1)

    return code_map.get(
        code,
        code
    )

importance["Description"] = (
    importance["Feature"]
    .apply(decode_feature)
)

# Display top predictors
display(
    importance[
        ["Description", "Posterior_Mean"]
    ].head(5)
)

# **Step 11: Feature Importance Chart**

In [ ]:
top = importance.head(10)

plt.figure(figsize=(8,4))

sns.barplot(
    data=top,
    x="Posterior_Mean",
    y="Description",
    hue="Description",
    palette="viridis",
    legend=False
)

plt.title(
    "Top Predictors of Supplier Risk"
)

plt.xlabel(
    "Posterior Mean Effect"
)

plt.ylabel(
    "Predictor"
)

plt.title(
    "Top Predictors of Supplier Risk"
)

plt.tight_layout()
plt.show()

# **Step 12: Risk Segmentation**

In [ ]:
risk_df = pd.DataFrame({
    "Risk_Probability": probability
})

risk_df["Risk_Level"] = pd.cut(
    risk_df["Risk_Probability"],
    bins=[0,.30,.70,1],
    labels=[
        "Low",
        "Medium",
        "High"
    ]
)

risk_df["Risk_Level"].value_counts()

# **Step 13: Risk Distribution**

In [ ]:
plt.figure(figsize=(6,4))

sns.histplot(
    probability,
    bins=20,
    kde=True
)

plt.title(
    "Supplier Risk Distribution"
)

plt.xlabel(
    "Risk Probability"
)

plt.show()